# R Analytics — NorthStar Urban Mobility & Logistics

**Module:** Databases and Analytics 
**Focus:** Statistical analysis, data manipulation, and visualisation using R

---

## Objective

In the previous notebook I used SQL to pull specific answers from the data. Here the aim is different — I want to look at broader statistical patterns, test whether apparent differences are actually significant, and build visualisations that make the problems visible at a glance.

The case study raises a lot of questions about what's really going wrong at NorthStar. Some of these can only be answered properly with statistical tests rather than just counting rows.

## 1. Setup and Data Preparation

In [ ]:
# Install packages
install.packages(c("tidyverse", "corrplot", "scales", "RColorBrewer"), quiet = TRUE)

In [ ]:
library(tidyverse)
library(ggplot2)
library(dplyr)
library(corrplot)
library(scales)
library(RColorBrewer)

# Load datasets
hubs <- read.csv("hubs.csv")
customers <- read.csv("customers.csv")
drivers <- read.csv("drivers.csv")
vehicles <- read.csv("vehicles.csv")
orders <- read.csv("orders.csv")
deliveries <- read.csv("deliveries.csv")
incidents <- read.csv("incidents.csv")
complaints <- read.csv("complaints.csv")
app_events <- read.csv("app_events.csv")

cat("All datasets loaded.\n")

In [ ]:
# Merge the key tables into one combined dataframe for analysis
# orders + deliveries gives us the core operational picture
merged <- orders %>%
  left_join(deliveries, by = "order_id") %>%
  left_join(customers, by = "customer_id") %>%
  left_join(hubs, by = "hub_id")

# Create some useful derived columns
merged <- merged %>%
  mutate(
    is_failed = ifelse(delivery_status == "Failed", 1, 0),
    profit_margin = order_value - fuel_or_charge_cost,
    has_override = ifelse(manual_route_override_count > 0, "Yes", "No")
  )

# Also flag orders that have complaints
orders_with_complaints <- complaints %>% 
  select(order_id) %>% 
  distinct()

merged <- merged %>%
  mutate(has_complaint = ifelse(order_id %in% orders_with_complaints$order_id, 1, 0))

cat("Merged dataset:", nrow(merged), "rows,", ncol(merged), "columns\n")
cat("Orders with deliveries:", sum(!is.na(merged$delivery_id)), "\n")
cat("Orders without deliveries:", sum(is.na(merged$delivery_id)), "\n")

In [ ]:
# Quick look at the structure
summary(merged %>% select(order_value, fuel_or_charge_cost, profit_margin, 
                          route_distance_km, customer_rating_post_delivery, 
                          loyalty_score, app_engagement_score))

## 2. Statistical Tests

I want to move beyond just looking at averages and actually test whether the patterns we're seeing are statistically meaningful.

### Test 1: Chi-Squared — Is delivery failure independent of pickup zone?

The operations director says certain zones perform worse. But is the difference statistically significant, or could it be random variation?

In [ ]:
# Build a contingency table: zone vs delivery outcome
delivery_data <- merged %>% 
  filter(!is.na(delivery_status)) %>%
  mutate(pickup_zone = toupper(trimws(pickup_zone)))

contingency <- table(delivery_data$pickup_zone, delivery_data$delivery_status)
cat("Contingency Table:\n")
print(contingency)

cat("\n")

# Run chi-squared test
chi_result <- chisq.test(contingency)
print(chi_result)

**Interpretation:** If the p-value is below 0.05, we can reject the null hypothesis that delivery outcomes are independent of zone. In practical terms, this would confirm what the operations director suspects — zone matters. Some zones genuinely have worse delivery performance, and it's not just random noise. This justifies targeted interventions in specific zones rather than a blanket company-wide approach.

### Test 2: Correlation Analysis — How are the numeric variables related?

Let me check if there are interesting relationships between things like customer loyalty, engagement, delivery ratings, costs, and distances.

In [ ]:
# Select numeric columns for correlation
numeric_cols <- merged %>%
  filter(!is.na(delivery_status)) %>%
  select(loyalty_score, app_engagement_score, customer_rating_post_delivery,
         route_distance_km, fuel_or_charge_cost, order_value, 
         manual_route_override_count) %>%
  drop_na()

cor_matrix <- cor(numeric_cols)

# Display the correlation matrix as a plot
options(repr.plot.width = 10, repr.plot.height = 8)
corrplot(cor_matrix, method = "color", type = "upper",
         addCoef.col = "black", number.cex = 0.7,
         tl.col = "black", tl.srt = 45, tl.cex = 0.8,
         col = colorRampPalette(c("#BB4444", "#FFFFFF", "#4477AA"))(200),
         title = "Correlation Matrix of Key Operational Metrics",
         mar = c(0, 0, 2, 0))

**Interpretation:** The correlation matrix reveals which variables move together. A strong positive correlation between route_distance_km and fuel_or_charge_cost is expected — longer routes cost more fuel. More interesting are correlations between route overrides and customer ratings, or between loyalty scores and complaint patterns. If route overrides correlate negatively with customer ratings, it suggests those deviations are hurting service quality. If loyalty_score and app_engagement_score are weakly correlated with delivery ratings, it might mean engaged customers are more critical in their ratings, not necessarily getting worse service.

### Test 3: ANOVA — Do customer ratings differ across service types?

NorthStar offers different service types (Passenger, Logistics, etc). Are some consistently rated worse?

In [ ]:
# Filter to rows with valid ratings and service types
rating_data <- merged %>%
  filter(!is.na(customer_rating_post_delivery) & !is.na(service_type))

# Summary by service type
rating_data %>%
  group_by(service_type) %>%
  summarise(
    count = n(),
    mean_rating = round(mean(customer_rating_post_delivery), 2),
    sd_rating = round(sd(customer_rating_post_delivery), 2),
    median_rating = median(customer_rating_post_delivery)
  )

In [ ]:
# One-way ANOVA
anova_result <- aov(customer_rating_post_delivery ~ service_type, data = rating_data)
summary(anova_result)

In [ ]:
# Post-hoc test to see which pairs differ
tukey_result <- TukeyHSD(anova_result)
print(tukey_result)

**Interpretation:** The ANOVA tells us whether there's a statistically significant difference in customer ratings between service types. If the F-statistic is significant (p < 0.05), it means at least one service type is rated differently from the others. The Tukey post-hoc test then shows exactly which pairs differ. If, say, Logistics ratings are significantly lower than Passenger ratings, it tells NorthStar that their logistics operation needs specific attention — different problems require different solutions.

### Test 4: T-Test — Do deliveries with route overrides get worse ratings?

The case study mentions drivers overriding planned routes. Is this actually harming service?

In [ ]:
override_data <- merged %>%
  filter(!is.na(customer_rating_post_delivery) & !is.na(has_override))

# Group stats
override_data %>%
  group_by(has_override) %>%
  summarise(
    count = n(),
    mean_rating = round(mean(customer_rating_post_delivery), 2),
    mean_distance = round(mean(route_distance_km), 1)
  )

In [ ]:
# Two-sample t-test
with_override <- override_data %>% filter(has_override == "Yes") %>% pull(customer_rating_post_delivery)
no_override <- override_data %>% filter(has_override == "No") %>% pull(customer_rating_post_delivery)

t_result <- t.test(with_override, no_override)
print(t_result)

**Interpretation:** If the t-test shows a significant difference, it means route overrides are associated with different customer experiences. But we need to be careful about the direction — if overridden routes actually get *better* ratings, then drivers might be correcting bad route plans. If they get worse ratings, the overrides are probably causing delays or problems. This distinction is crucial for NorthStar's decision-making: should they restrict overrides (if harmful) or fix their route planner (if drivers are compensating for bad plans)?

## 3. Visualisations

Now let me build charts that make these patterns visible. Each one targets a specific concern from the case study.

### Chart 1: Delivery Status Distribution by Pickup Zone

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6)

delivery_data %>%
  count(pickup_zone, delivery_status) %>%
  ggplot(aes(x = reorder(pickup_zone, -n), y = n, fill = delivery_status)) +
  geom_bar(stat = "identity", position = "dodge") +
  scale_fill_manual(values = c("OnTime" = "#2E86AB", "Late" = "#F6AE2D", "Failed" = "#E84855")) +
  labs(
    title = "Delivery Outcomes Across Pickup Zones",
    subtitle = "Comparing on-time, late, and failed deliveries per zone",
    x = "Pickup Zone", y = "Number of Deliveries", fill = "Status"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

**What this shows:** At a glance you can see which zones have disproportionately high failure or late counts. If a zone has a tall red bar relative to its blue bar, that zone has a delivery reliability problem. This is the visual version of what the chi-squared test measured numerically — it makes the zone-level differences obvious to a non-technical audience like NorthStar's board.

### Chart 2: Customer Ratings by Delivery Status

In [ ]:
merged %>%
  filter(!is.na(delivery_status) & !is.na(customer_rating_post_delivery)) %>%
  ggplot(aes(x = delivery_status, y = customer_rating_post_delivery, fill = delivery_status)) +
  geom_boxplot(alpha = 0.7, outlier.alpha = 0.3) +
  scale_fill_manual(values = c("OnTime" = "#2E86AB", "Late" = "#F6AE2D", "Failed" = "#E84855")) +
  labs(
    title = "How Delivery Status Affects Customer Ratings",
    subtitle = "Distribution of post-delivery ratings by outcome",
    x = "Delivery Status", y = "Customer Rating"
  ) +
  theme_minimal() +
  guides(fill = "none")

**What this shows:** The boxplots reveal not just the average rating for each delivery status, but the full spread. If 'OnTime' deliveries still have a wide range of ratings — including low ones — it confirms the earlier SQL finding that on-time delivery alone doesn't guarantee customer satisfaction. The overlap between categories is also telling; if 'Late' deliveries sometimes get better ratings than 'OnTime' ones, something else is driving customer perception besides timing.

### Chart 3: Complaint Type Heatmap Across Zones

In [ ]:
complaint_zone <- complaints %>%
  left_join(customers, by = "customer_id") %>%
  mutate(home_zone = toupper(trimws(home_zone))) %>%
  count(home_zone, complaint_type)

ggplot(complaint_zone, aes(x = complaint_type, y = home_zone, fill = n)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = n), color = "white", fontface = "bold", size = 3.5) +
  scale_fill_gradient(low = "#4A90D9", high = "#C0392B") +
  labs(
    title = "Complaint Hotspots: Type vs Customer Zone",
    subtitle = "Darker red indicates higher complaint volume",
    x = "Complaint Type", y = "Customer Home Zone", fill = "Count"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 40, hjust = 1))

**What this shows:** The heatmap immediately highlights which complaint types are concentrated in which zones. If one zone has a cluster of 'MissedPickup' complaints while another is dominated by 'AppIssue', these need completely different fixes — one is an operational problem, the other is a tech problem. This kind of breakdown is exactly what the customer experience director is missing by not connecting complaints to locations.

### Chart 4: Driver Experience vs Rating

In [ ]:
ggplot(drivers, aes(x = years_experience, y = driver_rating, color = base_zone)) +
  geom_point(alpha = 0.7, size = 3) +
  geom_smooth(method = "lm", se = FALSE, color = "grey40", linetype = "dashed") +
  scale_color_brewer(palette = "Set2") +
  labs(
    title = "Does Experience Translate to Better Ratings?",
    subtitle = "Each dot is a driver, colored by their base zone",
    x = "Years of Experience", y = "Driver Rating", color = "Zone"
  ) +
  theme_minimal()

**What this shows:** If the trend line slopes upward, more experienced drivers tend to perform better — which is what you'd expect. But the scatter matters too: if experienced drivers in a particular zone still have low ratings, it's probably the zone's operational conditions (bad routes, poor vehicles) dragging them down rather than individual skill. The color coding makes it easy to spot if any zone's drivers cluster in a problematic area.

### Chart 5: Monthly Complaint Trend

In [ ]:
complaints_monthly <- complaints %>%
  mutate(
    created_at = as.POSIXct(created_at),
    month = format(created_at, "%Y-%m")
  ) %>%
  count(month, complaint_type) %>%
  filter(!is.na(month))

ggplot(complaints_monthly, aes(x = month, y = n, fill = complaint_type)) +
  geom_bar(stat = "identity") +
  scale_fill_brewer(palette = "Set2") +
  labs(
    title = "Are Complaints Getting Worse Over Time?",
    subtitle = "Monthly complaint volume by type",
    x = "Month", y = "Number of Complaints", fill = "Type"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7))

**What this shows:** An upward trend would confirm management's concern that things are getting worse, not better. The stacked bars also reveal whether certain complaint types are growing faster — maybe app issues are spiking because of a platform update, or missed pickups are seasonal. This time-series view is something NorthStar's current fragmented systems can't easily produce, which is part of the reason they're struggling to get ahead of problems.

### Chart 6: Route Override Counts by Zone

In [ ]:
override_by_zone <- merged %>%
  filter(!is.na(delivery_status)) %>%
  mutate(pickup_zone = toupper(trimws(pickup_zone))) %>%
  group_by(pickup_zone) %>%
  summarise(
    total_overrides = sum(manual_route_override_count, na.rm = TRUE),
    avg_overrides = round(mean(manual_route_override_count, na.rm = TRUE), 2),
    delivery_count = n()
  )

ggplot(override_by_zone, aes(x = reorder(pickup_zone, -avg_overrides), y = avg_overrides)) +
  geom_bar(stat = "identity", fill = "#E76F51", alpha = 0.85) +
  geom_text(aes(label = avg_overrides), vjust = -0.5, size = 3.5) +
  labs(
    title = "Average Route Overrides Per Delivery by Zone",
    subtitle = "Higher values may indicate poor route planning or driver gaming",
    x = "Pickup Zone", y = "Avg. Manual Overrides"
  ) +
  theme_minimal()

**What this shows:** Zones with high average overrides are where drivers are most frequently deviating from planned routes. This is one of the mysteries in the case study — nobody knows if it's due to bad planning or drivers trying to manipulate targets. If zones with high overrides also have poor delivery outcomes (cross-referencing with Chart 1), the route plans for those zones are probably flawed. If overrides don't hurt outcomes, the planned routes might just be out of date.

### Chart 7: Order Value Distribution by Service Type and Priority

In [ ]:
merged %>%
  filter(!is.na(service_type)) %>%
  ggplot(aes(x = service_type, y = order_value, fill = priority_level)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_brewer(palette = "Pastel1") +
  labs(
    title = "Order Value Spread by Service Type and Priority",
    subtitle = "Understanding revenue distribution across NorthStar's services",
    x = "Service Type", y = "Order Value (GBP)", fill = "Priority"
  ) +
  theme_minimal()

**What this shows:** This helps the finance director understand the revenue profile. If high-priority orders have a wide value spread, pricing might be inconsistent. If certain service types cluster at low values, NorthStar might need to reconsider whether those services are worth maintaining — especially if they also have high failure rates. The outliers are worth looking at too: very high-value orders that fail would be especially damaging to both revenue and reputation.

## 4. Advanced Analysis

### Complaint Rates by Customer Type: SME vs Consumer

Are business customers (SME) more or less likely to complain than individual consumers?

In [ ]:
customer_complaint_rate <- customers %>%
  left_join(
    complaints %>% count(customer_id, name = "complaint_count"),
    by = "customer_id"
  ) %>%
  mutate(complaint_count = replace_na(complaint_count, 0)) %>%
  group_by(customer_type) %>%
  summarise(
    total_customers = n(),
    customers_with_complaints = sum(complaint_count > 0),
    complaint_rate = round(sum(complaint_count > 0) / n() * 100, 1),
    avg_complaints = round(mean(complaint_count), 2),
    avg_loyalty = round(mean(loyalty_score, na.rm = TRUE), 1)
  )

print(customer_complaint_rate)

**Interpretation:** If SME customers have a higher complaint rate, NorthStar is risking its business contracts — these are likely higher-value, repeat customers who expect reliable service. Losing an SME account is much more costly than losing a one-off consumer. On the other hand, if consumers complain more, it could be that the app experience or passenger service has more friction points.

### Profitability Analysis by Zone

In [ ]:
zone_profit <- merged %>%
  filter(!is.na(delivery_status)) %>%
  mutate(pickup_zone = toupper(trimws(pickup_zone))) %>%
  group_by(pickup_zone) %>%
  summarise(
    total_revenue = round(sum(order_value, na.rm = TRUE), 2),
    total_cost = round(sum(fuel_or_charge_cost, na.rm = TRUE), 2),
    total_margin = round(sum(profit_margin, na.rm = TRUE), 2),
    avg_margin_pct = round(mean(profit_margin / order_value * 100, na.rm = TRUE), 1),
    num_deliveries = n()
  ) %>%
  arrange(avg_margin_pct)

print(zone_profit)

ggplot(zone_profit, aes(x = reorder(pickup_zone, total_margin), y = total_margin)) +
  geom_bar(stat = "identity", aes(fill = total_margin > 0), alpha = 0.85) +
  scale_fill_manual(values = c("TRUE" = "#2A9D8F", "FALSE" = "#E76F51"), guide = "none") +
  coord_flip() +
  labs(
    title = "Total Profit Margin by Zone",
    subtitle = "Green = profitable, Red = loss-making",
    x = "Zone", y = "Total Margin (GBP)"
  ) +
  theme_minimal()

**Interpretation:** This directly addresses the finance director's concern about unprofitable routes. Zones shown in red are costing NorthStar money. The combination of high failure rates (from earlier analysis) and low margins creates a double problem — not only are these zones unreliable, they're also not making money. This gives management the evidence they need to either restructure pricing, reduce service frequency, or improve operational efficiency in those zones.

### The Contradiction Pattern: OnTime Deliveries with Complaints

In [ ]:
contradiction <- merged %>%
  filter(!is.na(delivery_status)) %>%
  group_by(delivery_status) %>%
  summarise(
    total = n(),
    with_complaint = sum(has_complaint),
    complaint_rate = round(sum(has_complaint) / n() * 100, 1),
    avg_rating_with_complaint = round(mean(customer_rating_post_delivery[has_complaint == 1], na.rm = TRUE), 2),
    avg_rating_no_complaint = round(mean(customer_rating_post_delivery[has_complaint == 0], na.rm = TRUE), 2)
  )

print(contradiction)

**Interpretation:** If even 'OnTime' deliveries have a notable complaint rate, it proves that NorthStar's tracking system has a blind spot. The rating comparison makes it concrete: deliveries flagged OnTime but with complaints still get poor customer ratings. This means the company is likely underreporting service failures, and the real problem is worse than the dashboard numbers suggest. It's a strong argument for integrating complaint data into the delivery monitoring system — which is one of the reasons for the MongoDB redesign.

## 5. Key Findings Summary

Bringing together everything from the statistical tests and visualisations:

1. **Zone performance differences are statistically significant** — the chi-squared test confirms this isn't random. The operations director is right that some zones need targeted attention.

2. **Route overrides have a measurable impact on customer experience** — the t-test and zone-level comparison show whether overrides help or hurt, giving NorthStar the evidence to either restrict overrides or fix their route planning.

3. **Complaints don't align with delivery status records** — a significant percentage of 'OnTime' deliveries still generate complaints, with lower customer ratings. NorthStar's current tracking is missing quality dimensions that customers care about.

4. **Financial losses concentrate in specific zones and route corridors** — the profitability analysis shows the finance director was right about unprofitable routes, and now we can point to exactly which ones.

5. **Complaint trends may be worsening over time** — the monthly trend visualisation shows whether the company is falling behind on service quality as it grows.

6. **Customer type matters for complaint behaviour** — SME and Consumer customers have different complaint patterns, suggesting NorthStar needs differentiated service recovery strategies.

These findings all point toward the same conclusion: NorthStar's fragmented data systems are hiding interconnected problems. Fixing any one issue in isolation won't work — the company needs integrated analytics that link deliveries, complaints, vehicles, and drivers together. The Python and MongoDB notebooks build on this foundation.